# Checking the output of the importer for several issues


## basic imports and setup

In [1]:
from collections import namedtuple
import json, os
from datetime import date
import pandas as pd
import requests
from bs4 import BeautifulSoup
from ast import literal_eval
from impresso_essentials.utils import get_provider_for_alias
from text_preparation.importers.classes import CanonicalPage

In [2]:
EDITIONS_MAPPINGS = {1: "a", 2: "b", 3: "c", 4: "d", 5: "e", 6: "f", 7: "g", 8: "h", 9: "i", 10: "j", 11: "k", 12: "l", 13: "m", 14: "n", 15: "o", 16: "p", 17: "q", 18: "r", 19: "s", 20: "t", 21: "u", 22: "v", 23: "w", 24: "x", 25: "y", 26: "z",
                     27: "aa", 28: "ab", 29: "ac", 30: "ad", 31: "ae", 32: "af", 33: "ag", 34: "ah", 35: "ai", 36: "aj", 37: "ak", 38: "al", 39: "am", 40: "an", 41: "ao", 42: "ap", 43: "aq", 44: "ar", 45: "as", 46: "at", 47: "au", 48: "av", 49: "aw", 50: "ax", 51: "ay", 52: "az"
                     }
BASE_DIR = "/mnt/project_impresso/original"
JSON_FILE = "/home/piconti/impresso-text-acquisition/text_preparation/data/issue_indices/issue_index.{prov}.json"


## Test issue and page

### Display functions

In [3]:
def dump_issue_structure(issue):
    print("\n================= ISSUE OVERVIEW =================")
    print("ID:", issue.id)
    print("Alias:", issue.alias)
    print("Date:", issue.date)
    print("Edition:", issue.edition)
    print("Path:", issue.path)
    print("ARK:", issue.ark_id)

    print("\n----------- IMAGE PROPERTIES (per page) ----------")
    for p, props in issue.image_properties.items():
        print(f"Page {p}: {props}")

    print("\n================= CONTENT ITEMS =================")
    for ci in issue.issue_data["i"]:
        print("\n-----------------------------------------------")
        print(f"CI ID: {ci['m']['id']}")
        print(f"  Type: {ci['m']['tp']}")
        if ci['m']['tp'] == 'image':
            if 'pOf' in ci:
                print(f"        Image is part of: {ci['pOf']}")
            else:
                print(f"        Image is not associated to a CI")
        
        if ci['m'].get('t'):
            print(f"  Title: {ci['m'].get('t')}")
        else: 
            print("There is no title")
        print(f"  Language: {ci['m'].get('lg')}")
        print(f"  Pages (pp): {ci['m']['pp']}")
        print(f"  Reading order index: {ci['m']['ro']}")

        if 'section_title' in ci:
            print(f"  Section {ci['section_title']['section_id']} title info:")
            print(f"    Section title text: {ci['section_title']['title_text']}")
            print(f"    Section CIs: {ci['section_title']['composing_ci_ids']}")
            print(f"    Section title parts: {ci['section_title']['heading_legacy_parts']}")

        print(f"  Legacy info:")
        print(f"    Legacy id: {ci['l']['id']}")
        if 'ark_id' in ci['l']:
            print(f"    Ark id: {ci['l']['ark_id']}")
        elif "ppn" in ci['l']:
            print(f"    issue PPN: {ci['l']['ppn']}")
            print(f"    title PPN: {ci['l']['title_ppn']}")
        print(f"    legacy source: {ci['l']['src_files']}")
        # Legacy OCR parts
        if "parts" in ci["l"]:
            print("  Legacy OCR parts:")
            for part in ci["l"]["parts"]:
                print(f"    - comp_id={part['comp_id']}, "
                      f"role={part['comp_role']}, "
                      f"fileid={part['comp_fileid']}, "
                      f"page={part['comp_page_no']}")

        if "canonical_parts" in ci["l"]:
            print("  Canonical sub-articles:", ci["l"]["canonical_parts"])


In [4]:
def find_page(issue, page_no: int = None, page_id: str = None):
    for p in issue.pages:        
        if page_no and p.number == page_no:
            page = p
            break
        if page_id and p.id == page_id:
            page = p
            break
    return page

In [5]:
def dump_page_structure(issue, page_no: int = None, page_id: str = None):
    """
    Dump exactly one page, printing *all* regions/paragraphs/lines/tokens.

    Provide either:
      - page_no (int): the physical page number (issue.pages[i].number), or
      - page_id (str): canonical page id like 'tageblatt-1950-11-11-a-p0001'

    Example:
        dump_one_page(luxIssue, page_no=1)
        dump_one_page(luxIssue, page_id="tageblatt-1950-11-11-a-p0001")
    """

    # Find the page object
    page = find_page(issue, page_no=page_no, page_id=page_id)
    
    # page = issue.pages[page_no] if page_no else getattr(issue.pages, _id)
    page.parse()
    page_data = page.page_data

    print("\n================= PAGE DUMP =================")
    print("Issue ID:", getattr(issue, "id", None))
    print("PAGE ID:", page_data.get("id", None))
    print("  Number:", page_data.get("number", None))

    fw = page_data.get("fw", None)
    fh = page_data.get("fh", None)
    print("  Dimensions (fw x fh):", f"{fw} x {fh}")

    print("  st:", page_data.get("st", None))
    print("  sm:", page_data.get("sm", None))
    if "cc" in page_data:
        print("  cc:", page_data.get("cc"))
    if "ts" in page_data:
        print("  ts:", page_data.get("ts"))
    if "cdt" in page_data:
        print("  cdt:", page_data.get("cdt"))

    regions = page_data.get("r", [])
    print("\n================= REGIONS =================")
    print("Regions count:", len(regions))

    for ridx, region in enumerate(regions, start=1):
        rc = region.get("c")
        rpof = region.get("pOf", None)
        paragraphs = region.get("p", []) or []

        print("\n-----------------------------------------------")
        print(f"Region {ridx}")
        print(f"  c: {rc}")
        if "pOf" in region:
            print(f"  pOf: {rpof}")
        print(f"  Paragraphs: {len(paragraphs)}")

        for pidx, para in enumerate(paragraphs, start=1):
            pc = para.get("c")
            lines = para.get("l", []) or []

            print(f"\n  Paragraph {pidx}")
            if pc is not None:
                print(f"    c: {pc}")
            print(f"    Lines: {len(lines)}")

            for lidx, line in enumerate(lines, start=1):
                lc = line.get("c")
                tokens = line.get("t", []) or []

                print(f"\n    Line {lidx}")
                print(f"      c: {lc}")
                print(f"      Tokens: {len(tokens)}")

                # print all tokens
                for tidx, tok in enumerate(tokens, start=1):
                    tx = tok.get("tx")
                    tc = tok.get("c")
                    s = tok.get("s", None)
                    gn = tok.get("gn", None)
                    hy = tok.get("hy", None)
                    nf = tok.get("nf", None)

                    extras = []
                    if s is not None:
                        extras.append(f"s={s}")
                    if gn is not None:
                        extras.append(f"gn={gn}")
                    if hy is not None:
                        extras.append(f"hy={hy}")
                    if nf is not None:
                        extras.append(f"nf={nf}")

                    extras_str = (" [" + ", ".join(extras) + "]") if extras else ""
                    print(f"        - {tidx:04d}: tx={tx!r}, c={tc}{extras_str}")


In [6]:
def show_smaller(img, max_width=900):
    w, h = img.size
    if w <= max_width:
        img.show()
        return img

    scale = max_width / w
    new_size = (int(w * scale), int(h * scale))
    img_small = img.resize(new_size, Image.Resampling.LANCZOS)
    img_small.show()
    return img_small

In [7]:
import os
from PIL import Image
from text_preparation.utils import draw_box_on_img

ATTACHED_WIDTH = 3
UNATTACHED_WIDTH = 12

def find_regions_per_ci(page_regions):
    ci_regions = {}
    sections = []
    unattached_counter = 1
    for region in page_regions:
        if 'section_pOf' in region:
            sections.append((region["section_pOf"], region))
            continue

        if "pOf" in region and region["pOf"]:
            og_ci_id = region["pOf"]
        else:
            og_ci_id = f"No attached CI {unattached_counter}"
            unattached_counter += 1

        ci_regions.setdefault(og_ci_id, []).append(region)
    return ci_regions, sections

# Reserve ONE specific color for unlinked regions
UNATTACHED_COLOR = "red"

colors = ["green", "blue", "purple", "orange", "cyan", "sienna", "limegreen", "pink"]
colors_attached = [c for c in colors if c != UNATTACHED_COLOR]

def coords_to_xy(c):
    """[x,y,w,h] -> [x1,y1,x2,y2]"""
    x, y, w, h = c
    return [x, y, x + w, y + h]

def color_for_ci(ci_id: str) -> str:
    """
    Stable mapping CI id -> color for ATTACHED regions.
    Uses the last 4 digits if present, otherwise hashes.
    never returns UNATTACHED_COLOR.
    """
    tail = ci_id[-4:]
    if tail.isdigit():
        return colors_attached[int(tail) % len(colors_attached)]
    return colors_attached[abs(hash(ci_id)) % len(colors_attached)]

def show_smaller(img, max_width=900):
    w, h = img.size
    if w <= max_width:
        img.show()
        return img
    scale = max_width / w
    new_size = (int(w * scale), int(h * scale))
    img_small = img.resize(new_size, Image.Resampling.LANCZOS)
    img_small.show()
    return img_small


def draw_page_boxes(
    page_object,
    issue,
    img_path: str,
    show=True,
    save_dir: str | None = None,
    draw_regions=True,
    draw_image_cis=True,
    color_to_print="all",   # "all" or a specific color string
    max_width=800
):
    print(f"\n=== Page: {page_object.id} (no={page_object.number}) ===")
    img = Image.open(img_path).convert("RGB")

    page_object.parse()
    page_regions = page_object.page_data.get("r", [])

    # Draw image content items (tp=image)
    all_img_region_coords_xy = set()
    if draw_image_cis:
        image_cis_on_page = [
            ci for ci in issue.issue_data["i"]
            if page_object.number in ci["m"].get("pp", []) and ci["m"].get("tp") == "image"
        ]
        for idx, ci in enumerate(image_cis_on_page):
            if "c" not in ci:
                continue
            ci_id = ci["m"]["id"]
            # attached colors (never red)
            ci_color = color_for_ci(ci.get("pOf") or ci_id)

            xy = coords_to_xy(ci["c"])
            all_img_region_coords_xy.add(tuple(xy))

            print(f"[IMG CI] {ci_id} pOf={ci.get('pOf')} -> {ci_color} coords={ci['c']}")
            if color_to_print == "all" or ci_color == color_to_print:
                img = draw_box_on_img(img_path, xy, img=img, color=ci_color, text=f"{ci_id}")

    # Draw OCR regions grouped by pOf
    if draw_regions:
        ci_regions_map, sections = find_regions_per_ci(page_regions)

        for (sec_cis, sec_region) in sections:
            if "c" not in sec_region or sec_region["c"] is None:
                continue
            xy = coords_to_xy(sec_region["c"])
            print(f"[TITLE of SECTION] comprised of CIs {sec_cis} -> darkred coords={sec_region['c']}")
            img = draw_box_on_img(
                img_path, xy, img=img, color='darkred', width=UNATTACHED_WIDTH, text=f"SECTION TITLE"
            )

        for ci_id, regs in ci_regions_map.items():
            is_unattached = ci_id.startswith("No attached CI")
            ci_color = UNATTACHED_COLOR if is_unattached else color_for_ci(ci_id)

            for r_idx, region in enumerate(regs):
                if "c" not in region or region["c"] is None:
                    continue
                xy = coords_to_xy(region["c"])

                # avoid drawing duplicates of image CI boxes
                if tuple(xy) in all_img_region_coords_xy:
                    continue

                print(f"[REG] {ci_id}-reg{r_idx} -> {ci_color} coords={region['c']}")
                if color_to_print == "all" or ci_color == color_to_print:
                    line_width = UNATTACHED_WIDTH if is_unattached else ATTACHED_WIDTH
                    img = draw_box_on_img(
                        img_path, xy, img=img, color=ci_color, width=line_width, text=f"{ci_id}-r{r_idx}"
                    )

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        out_path = os.path.join(save_dir, f"{page_object.id}.jpg")
        img.save(out_path, quality=90)
        print(f"Saved -> {out_path}")

    if show:
        show_smaller(img, max_width=max_width)

    return img


def draw_issue_boxes(
    issue,
    images_dir: str,
    save_dir: str | None = None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 800
):
    for page in sorted(issue.pages, key=lambda x: x.number):
        if hasattr(issue, 'image_files_by_number'):
            tif_name = issue.image_files_by_number.get(page.number)
        else:
            tif_name = page.filename.replace('xml', 'tif')

        if tif_name is None:
            print(f"WARNING: no image for page {page.number} ({page.id})")
            continue

        img_path = os.path.join(images_dir, tif_name)
        if not os.path.exists(img_path):
            print(f"WARNING: missing image file: {img_path}")
            continue

        draw_page_boxes(
            page_object=page,
            issue=issue,
            img_path=img_path,
            show=show,
            save_dir=save_dir,
            draw_regions=draw_regions,
            draw_image_cis=draw_image_cis,
            color_to_print="all",
            max_width=max_width
        )


In [8]:
def get_issue_from_id(issue_id, all_entries, entry2issue_func, IssueClass):
    alias, year, month, day, edition = issue_id.split('-')
    entry = list(filter(lambda x: x['day']==day and x['edition']==edition, all_entries[alias].get(year)[month]))[0]

    print(f"entry for issue {issue_id}: {entry}")
    issuedir = entry2issue_func(alias, year, month, entry, BASE_DIR)
    print(f"issuedir for {issue_id}: {issuedir}")

    return IssueClass(issuedir)

## BNL DATA

In [8]:
from text_preparation.importers.lux.helpers import convert_coordinates, div_has_body_or_article, find_section_articles
from text_preparation.importers.lux.classes import LuxNewspaperPage, LuxNewspaperIssue
from text_preparation.importers.lux.detect import entry2issue as bnl_entry2issue


bnl_json = JSON_FILE.format(prov='bnl')

with open(bnl_json, "r") as f:
    all_bnl_entries = json.load(f)

### Tested issues

#### 1. Luxwort: `luxwort-1866-10-21-a`

In [ ]:
issue_id = "luxwort-1866-10-21-a"
alias = "luxwort"
year = "1866"
month = '10'


entry = list(filter(lambda x: x['day']=='21', all_entries[alias].get(year)[month]))[0]
print(f"entry: {entry}")

issuedir = entry2issue(alias, year, month, entry, BASE_DIR)
issuedir

In [ ]:
luxwort_1866_issue = LuxNewspaperIssue(issuedir)
dump_issue_structure(luxwort_1866_issue)

In [ ]:
dump_page_structure(luxwort_1866_issue, 1)

In [ ]:
images_dir = os.path.join(BASE_DIR, luxwort_1866_issue.path, 'images')

draw_issue_boxes(
    issue=luxwort_1866_issue,
    images_dir=images_dir,
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 600
)

#### 2. Luxwort: `luxwort-1890-04-06-a`

In [11]:
luxwort_id_2 = "luxwort-1890-04-05-b"
luxwort_issue_2 = get_issue_from_id(luxwort_id_2, all_bnl_entries, bnl_entry2issue, LuxNewspaperIssue)

entry for issue luxwort-1890-04-05-b: {'day': '05', 'edition': 'b', 'local_path': '/BNL/protected_015/964655_newspaper_luxwort_1890-04-05_02'}
issuedir for luxwort-1890-04-05-b: IssueDirectory(provider='BNL', alias='luxwort', date=datetime.date(1890, 4, 5), edition='b', path='/mnt/project_impresso/original/BNL/protected_015/964655_newspaper_luxwort_1890-04-05_02')


luxwort-1890-04-05-b - Problem when parsing the title of section MODSMD_SECTION6: item_title=None


luxwort-1890-04-05-b - Problem when parsing the title of section MODSMD_SECTION6: item_title=None


In [ ]:
dump_issue_structure(luxwort_issue_2)

In [ ]:
images_dir_luxwort_2 = os.path.join(BASE_DIR, luxwort_issue_2.path, 'images')

draw_issue_boxes(
    issue=luxwort_issue_2,
    images_dir=images_dir_luxwort_2,
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 600
)

In [ ]:
mets_doc = luxwort_issue_2.xml
sections = mets_doc.findAll("dmdSec")

# sort based on the ID string to pinpoint the generated canonical IDs
sections = sorted(sections, key=lambda elem: elem.get("ID").split("_")[1])

for section in sections:
    section_id = section.get("ID")
    if "ARTICLE" in section_id or "PICT" in section_id:
        parent_div = section.parent
        grand_parent_div = parent_div.parent
        print(f"section {section_id}: KEPT")
        #print(f"section {section_id}: parent_div type= {parent_div.get('TYPE')}, grand_parent_div_type={grand_parent_div.get('TYPE')}")
    elif "SECT" in section_id and mets_doc.find("div", {"DMDID": section_id}) is not None:
        div = mets_doc.find("div", {"DMDID": section_id})
        print(f"section {section_id}: reprocessed as part of _parse_sections:")
        for i in div.findChildren("div", recursive=False):
            print(f"   {section_id} - child ID: {i.get('DMDID')} type: {i.get('TYPE')}")
    else:
        print(f"section {section_id}: IGNORED")

#### 3. waeschfra: `waeschfra-1873-07-19-a`

In [ ]:
waeschfra_id = "waeschfra-1873-07-19-a"
waeschfra_issue = get_issue_from_id(waeschfra_id)

In [ ]:
dump_issue_structure(waeschfra_issue)

In [ ]:
images_dir_waeschfra = os.path.join(BASE_DIR, waeschfra_issue.path, 'images')

draw_issue_boxes(
    issue=waeschfra_issue,
    images_dir=images_dir_waeschfra,
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 600
)

#### 4. floreal: `floreal-1907-07-21-a`

In [ ]:
floreal_id = "floreal-1907-07-21-a"

floreal_issue = get_issue_from_id(floreal_id)

dump_issue_structure(floreal_issue)

In [ ]:
draw_issue_boxes(
    issue=floreal_issue,
    images_dir=os.path.join(BASE_DIR, floreal_issue.path, 'images'),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 500
)

In [ ]:
dump_page_structure(floreal_issue, 69)

In [ ]:
sorted_pages = sorted(floreal_issue.pages, key=lambda x: x.number)
sorted_pages[44].number, sorted_pages[44].page_data["iiif_img_base_uri"]

#### 5. dossierinformatioundokumentatioun: `dossierinformatioundokumentatioun-1996-06-15-a`

In [ ]:
dossierinformatioundokumentatioun_id = "dossierinformatioundokumentatioun-1996-06-15-a"

dossierinformatioundokumentatioun_issue = get_issue_from_id(dossierinformatioundokumentatioun_id)

dump_issue_structure(dossierinformatioundokumentatioun_issue)

In [ ]:
draw_issue_boxes(
    issue=dossierinformatioundokumentatioun_issue,
    images_dir=os.path.join(BASE_DIR, dossierinformatioundokumentatioun_issue.path, 'images'),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 300
)

#### 6. freihet: `freihet-1946-09-01-a`

In [ ]:
freihet_id = "freihet-1946-09-01-a"

freihet_issue = get_issue_from_id(freihet_id)

In [ ]:
mets_doc = freihet_issue.xml

eg_ci = {'m': 
         {'id': 'freihet-1946-09-01-a-i0086', 'pp': [1], 'tp': 'image', 't': 'Albert Wingert.', 'lg': 'de', 
          'iiif_link': 'https://iiif.eluxemburgensia.lu/image/iiif/2/ark:70795%2f7bcttgt2n0%2fpages%2f1/info.json'}, 
        'l': {'id': 'MODSMD_PICT2', 
              'parts': [{'comp_role': 'image', 'comp_id': 'ART3-5', 'comp_fileid': 'ALTO00001', 'comp_page_no': 1}, 
                        {'comp_role': 'caption', 'comp_id': 'ART3-6', 'comp_fileid': 'ALTO00001', 'comp_page_no': 1}], 
              'ark_id': 'ark:70795/7bcttgt2n0', 
              'src_files': {'mets_xml': '1946-09-01_01-mets.xml', 'alto_xml': [], 'image_tif': []}}, 
        'pOf': 'MODSMD_SECTION1', 'c': [88, 1690, 220, 297]}

In [ ]:
sections = mets_doc.findAll("dmdSec")

# sort based on the ID string to pinpoint the generated canonical IDs
sections = sorted(sections, key=lambda elem: elem.get("ID").split("_")[1])

for section in sections:
    section_id = section.get("ID")
    if "ARTICLE" in section_id or "PICT" in section_id:
        parent_div = section.parent
        grand_parent_div = parent_div.parent
        print(f"section {section_id}: KEPT")
        #print(f"section {section_id}: parent_div type= {parent_div.get('TYPE')}, grand_parent_div_type={grand_parent_div.get('TYPE')}")
    elif "SECT" in section_id and mets_doc.find("div", {"DMDID": section_id}) is not None:
        div = mets_doc.find("div", {"DMDID": section_id})
        has_body, has_article = div_has_body_or_article(div)
        if has_body:
            print(f"section {section_id}: RECONSTRUCTED into 1 CI - has_body={has_body}, has_article={has_article}:")
        if has_article and not has_body:
            print(f"section {section_id}: KEPT AS INDIVIDUAL CIs - has_body={has_body}, has_article={has_article}:")
            heading_kids = div.findChildren("div", {"TYPE": 'HEADING'}, recursive=False)
            heading_kids_parts = freihet_issue._parse_mets_div(heading_kids[0])
            print(f"   {section_id} - heading_kids parts: {heading_kids_parts}")
            for i in div.findChildren("div", recursive=False):
                if i.get("TYPE").lower() == 'heading':
                    parts = freihet_issue._parse_mets_div(i)
                    print(f"   {section_id} - HEADING child ID: {i.get('DMDID')} type: {i.get('TYPE')}: parts= {parts}")
            title_elements = section.find_all(
                    lambda tag: tag.name.endswith("titleInfo")
                    )
            titles = [
                ti.get_text(" ", strip=True)
                for ti in title_elements
                if ti.get_text(strip=True)
            ]
            # if only one, take it
            if len(titles) == 1:
                sect_title = titles[0]
            # if more than 2, join them
            elif len(titles) >= 2:
                first, second = titles[0], titles[1]
                # heuristic: short first title = rubric
                if len(first.split()) <= 2:
                    sect_title = f"{first} : {second}"
                else:
                    sect_title = " — ".join(titles)

            print(f"    {section_id} - section title: {sect_title}")


            #old_cis = find_section_articles(div, content_items)
    #else:
    #    print(f"section {section_id}: IGNORED")

In [ ]:
page = [p for p in freihet_issue.pages if p.number==1][0]
page_xml = page.xml

composed_block = page_xml.find("ComposedBlock", {"ID": "ART3-5"})
composed_block

In [ ]:
item_div = mets_doc.find_all("div", {"DMDID": eg_ci["l"]["id"]})
item_div

In [ ]:
dump_issue_structure(freihet_issue)

In [ ]:
dump_page_structure(freihet_issue, 5)

In [ ]:
freihet_issue.pages[0].page_data["iiif_img_base_uri"]

In [ ]:
draw_issue_boxes(
    issue=freihet_issue,
    images_dir=os.path.join(BASE_DIR, freihet_issue.path, 'images'),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 500
)

#### 7. tageblatt: `tageblatt-1950-11-11-a`

In [ ]:
tageblatt_id = "tageblatt-1950-11-11-a"

tageblatt_issue = get_issue_from_id(tageblatt_id)

In [ ]:
dump_issue_structure(tageblatt_issue)

In [ ]:
draw_issue_boxes(
    issue=tageblatt_issue,
    images_dir=os.path.join(BASE_DIR, tageblatt_issue.path, 'images'),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 600
)

#### 8. tageblatt: `tageblatt-1947-01-02-a`

In [ ]:
tageblatt2_id = "tageblatt-1947-01-02-a"

tageblatt2_issue = get_issue_from_id(tageblatt2_id)

dump_issue_structure(tageblatt2_issue)

In [ ]:
draw_issue_boxes(
    issue=tageblatt2_issue,
    images_dir=os.path.join(BASE_DIR, tageblatt2_issue.path, 'images'),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 500
)

#### 9. memoriala: `memoriala-1819-01-30-a`

In [ ]:
memoriala_id = "memoriala-1819-01-30-a"

memoriala_issue = get_issue_from_id(memoriala_id)

dump_issue_structure(memoriala_issue)

In [ ]:
draw_issue_boxes(
    issue=memoriala_issue,
    images_dir=os.path.join(BASE_DIR, memoriala_issue.path, 'images'),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 300
)

#### 10. gewerkschaftler: `gewerkschaftler-1918-02-16-a`

In [ ]:
gewerkschaftler_id = "gewerkschaftler-1918-02-16-a"

gewerkschaftler_issue = get_issue_from_id(gewerkschaftler_id)

dump_issue_structure(gewerkschaftler_issue)

In [ ]:
draw_issue_boxes(
    issue=gewerkschaftler_issue,
    images_dir=os.path.join(BASE_DIR, gewerkschaftler_issue.path, 'images'),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 400
)

#### 11. memorialrecueilspecial: `memorialrecueilspecial-1939-03-04-a`

In [ ]:
memorialrecueilspecial_id = "memorialrecueilspecial-1939-03-04-a"

memorialrecueilspecial_issue = get_issue_from_id(memorialrecueilspecial_id)

dump_issue_structure(memorialrecueilspecial_issue)

In [ ]:
draw_issue_boxes(
    issue=memorialrecueilspecial_issue,
    images_dir=os.path.join(BASE_DIR, memorialrecueilspecial_issue.path, 'images'),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 300
)

#### 12. jonghemecht: `jonghemecht-1933-03-01-a`

In [ ]:
jonghemecht_id = "jonghemecht-1933-03-01-a"

jonghemecht_issue = get_issue_from_id(jonghemecht_id)

dump_issue_structure(jonghemecht_issue)

In [20]:
draw_issue_boxes(
    issue=jonghemecht_issue,
    images_dir=os.path.join(BASE_DIR, jonghemecht_issue.path, 'images'),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 500
)

#### 13. jonghemecht: `jonghemecht-1927-02-01-a`

In [ ]:
jonghemecht_2_id = "jonghemecht-1927-02-01-a"

jonghemecht_2_issue = get_issue_from_id(jonghemecht_2_id)

dump_issue_structure(jonghemecht_2_issue)

In [ ]:
draw_issue_boxes(
    issue=jonghemecht_2_issue,
    images_dir=os.path.join(BASE_DIR, jonghemecht_2_issue.path, 'images'),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 300
)

#### 14. keisecker: `keisecker-1971-04-01-a`

In [ ]:
keisecker_id = "keisecker-1971-04-01-a"

keisecker_issue = get_issue_from_id(keisecker_id)

dump_issue_structure(keisecker_issue)

In [ ]:
draw_issue_boxes(
    issue=keisecker_issue,
    images_dir=os.path.join(BASE_DIR, keisecker_issue.path, 'images'),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 500
)

#### 15. keisecker: `keisecker-1980-03-15-a`

In [ ]:
keisecker_2_id = "keisecker-1980-03-15-a"

keisecker_2_issue = get_issue_from_id(keisecker_2_id)

dump_issue_structure(keisecker_2_issue)

In [ ]:
draw_issue_boxes(
    issue=keisecker_2_issue,
    images_dir=os.path.join(BASE_DIR, keisecker_2_issue.path, 'images'),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 500
)

#### 16. indeplux: `indeplux-1908-01-04-a`

In [ ]:
indeplux_id = "indeplux-1908-01-04-a"

indeplux_issue = get_issue_from_id(indeplux_id)

dump_issue_structure(indeplux_issue)

In [ ]:
draw_issue_boxes(
    issue=indeplux_issue,
    images_dir=os.path.join(BASE_DIR, indeplux_issue.path, 'images'),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 500
)

#### 17. indeplux: `indeplux-1921-04-01-a`

In [ ]:
indeplux_2_id = "indeplux-1921-04-01-a"

indeplux_2_issue = get_issue_from_id(indeplux_2_id)

dump_issue_structure(indeplux_2_issue)

In [ ]:
draw_issue_boxes(
    issue=indeplux_2_issue,
    images_dir=os.path.join(BASE_DIR, indeplux_2_issue.path, 'images'),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 500
)

#### 18. ordo: `ordo-1843-01-01-a`

In [ ]:
ordo_id = "ordo-1843-01-01-a"

ordo_issue = get_issue_from_id(ordo_id)

dump_issue_structure(ordo_issue)

In [ ]:
draw_issue_boxes(
    issue=ordo_issue,
    images_dir=os.path.join(BASE_DIR, ordo_issue.path, 'images'),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 500
)

#### 19. omnibus: `omnibus-1870-07-21-a`

In [ ]:
omnibus_id = "omnibus-1870-07-21-a"

omnibus_issue = get_issue_from_id(omnibus_id)

dump_issue_structure(omnibus_issue)

In [ ]:
draw_issue_boxes(
    issue=omnibus_issue,
    images_dir=os.path.join(BASE_DIR, omnibus_issue.path, 'images'),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 500
)

#### 20. kulturkampf: `kulturkampf-1875-04-24-a`

In [ ]:
kulturkampf_id = "kulturkampf-1875-04-24-a"

kulturkampf_issue = get_issue_from_id(kulturkampf_id)

dump_issue_structure(kulturkampf_issue)

In [ ]:
draw_issue_boxes(
    issue=kulturkampf_issue,
    images_dir=os.path.join(BASE_DIR, kulturkampf_issue.path, 'images'),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 500
)

#### 21. luxillustrierte: `luxillustrierte-1925-01-21-a`

In [ ]:
luxillustrierte_id = "luxillustrierte-1925-01-21-a"

luxillustrierte_issue = get_issue_from_id(luxillustrierte_id)

dump_issue_structure(luxillustrierte_issue)

In [ ]:
draw_issue_boxes(
    issue=luxillustrierte_issue,
    images_dir=os.path.join(BASE_DIR, luxillustrierte_issue.path, 'images'),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 500
)

#### 22. obermosel: `obermosel-1916-10-20-a`

In [ ]:
obermosel_id = "obermosel-1916-10-20-a"

obermosel_issue = get_issue_from_id(obermosel_id)

dump_issue_structure(obermosel_issue)

In [ ]:
draw_issue_boxes(
    issue=obermosel_issue,
    images_dir=os.path.join(BASE_DIR, obermosel_issue.path, 'images'),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 500
)

#### 23. revue: `revue-1947-02-16-a`

In [ ]:
revue_id = "revue-1947-02-16-a"

revue_issue = get_issue_from_id(revue_id)

dump_issue_structure(revue_issue)

In [ ]:
draw_issue_boxes(
    issue=revue_issue,
    images_dir=os.path.join(BASE_DIR, revue_issue.path, 'images'),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 500
)

#### 24. charribarri: `charribarri-1938-02-25-a`

In [ ]:
charribarri_id = "charribarri-1938-02-25-a"

charribarri_issue = get_issue_from_id(charribarri_id)

dump_issue_structure(charribarri_issue)

In [ ]:
draw_issue_boxes(
    issue=charribarri_issue,
    images_dir=os.path.join(BASE_DIR, charribarri_issue.path, 'images'),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 500
)

## SUB DATA

In [9]:
#from text_preparation.importers.sub.helpers import convert_coordinates, div_has_body_or_article, find_section_articles
from text_preparation.importers.sub.classes import SubNewspaperPage, SubNewspaperIssue
from text_preparation.importers.sub.detect import entry2issue as sub_entry2issue


sub_json = JSON_FILE.format(prov='sub')

with open(sub_json, "r") as f:
    all_sub_entries = json.load(f)

In [10]:
all_sub_entries

{'hamb_echo': {'1887': {'10': [{'day': '02',
     'edition': 'a',
     'local_path': '/SUB/Hamburger_Echo/1887/10/02/Ausgabe',
     'imgs_subdir': '',
     'imgs_ext': '.tif'},
    {'day': '04',
     'edition': 'a',
     'local_path': '/SUB/Hamburger_Echo/1887/10/04/Ausgabe',
     'imgs_subdir': '',
     'imgs_ext': '.tif'},
    {'day': '05',
     'edition': 'a',
     'local_path': '/SUB/Hamburger_Echo/1887/10/05/Ausgabe',
     'imgs_subdir': '',
     'imgs_ext': '.tif'},
    {'day': '06',
     'edition': 'a',
     'local_path': '/SUB/Hamburger_Echo/1887/10/06/Ausgabe',
     'imgs_subdir': '',
     'imgs_ext': '.tif'},
    {'day': '07',
     'edition': 'a',
     'local_path': '/SUB/Hamburger_Echo/1887/10/07/Ausgabe',
     'imgs_subdir': '',
     'imgs_ext': '.tif'},
    {'day': '08',
     'edition': 'a',
     'local_path': '/SUB/Hamburger_Echo/1887/10/08/Ausgabe',
     'imgs_subdir': '',
     'imgs_ext': '.tif'},
    {'day': '09',
     'edition': 'a',
     'local_path': '/SUB/Hamburger

### Tested Issues

#### 1. hamb_echo: `hamb_echo-1887-10-07-a`

In [12]:
id_1 = "hamb_echo-1887-10-07-a"

issue_1 = get_issue_from_id(id_1, all_sub_entries, sub_entry2issue, SubNewspaperIssue)

for p in issue_1.pages:
    p.add_issue(issue_1)

entry for issue hamb_echo-1887-10-07-a: {'day': '07', 'edition': 'a', 'local_path': '/SUB/Hamburger_Echo/1887/10/07/Ausgabe', 'imgs_subdir': '', 'imgs_ext': '.tif'}
issuedir for hamb_echo-1887-10-07-a: IssueDirectory(provider='SUB', alias='hamb_echo', date=datetime.date(1887, 10, 7), edition='a', path='/mnt/project_impresso/original/SUB/Hamburger_Echo/1887/10/07/Ausgabe')
hamb_echo-1887-10-07-a - adding page with filename 00000001.xml and file_id FILE_0001_FULLTEXT
hamb_echo-1887-10-07-a - adding page with filename 00000002.xml and file_id FILE_0002_FULLTEXT
hamb_echo-1887-10-07-a - adding page with filename 00000003.xml and file_id FILE_0003_FULLTEXT
hamb_echo-1887-10-07-a - adding page with filename 00000004.xml and file_id FILE_0004_FULLTEXT
hamb_echo-1887-10-07-a - adding page with filename 00000005.xml and file_id FILE_0005_FULLTEXT
hamb_echo-1887-10-07-a - adding page with filename 00000006.xml and file_id FILE_0006_FULLTEXT
hamb_echo-1887-10-07-a - adding page with filename 0000

In [13]:
dump_issue_structure(issue_1)


================= ISSUE OVERVIEW =================
ID: hamb_echo-1887-10-07-a
Alias: hamb_echo
Date: 1887-10-07
Edition: a
Path: /mnt/project_impresso/original/SUB/Hamburger_Echo/1887/10/07/Ausgabe
ARK: None

----------- IMAGE PROPERTIES (per page) ----------

================= CONTENT ITEMS =================

-----------------------------------------------
CI ID: hamb_echo-1887-10-07-a-i0001
  Type: page
There is no title
  Language: de
  Pages (pp): [1]
  Reading order index: 1
  Legacy info:
    Legacy id: PPN1754726119-PPN1754726119-00000001.xml
    issue PPN: PPN1754726119_18871007
    title PPN: PPN1754726119
    legacy source: {'mets_xml': 'PPN1754726119_18871007.xml', 'alto_xml': '00000001.xml', 'image_tif': '00000001.tif'}
  Legacy OCR parts:
    - comp_id=Page1_Block14, role=body, fileid=FILE_0001_FULLTEXT, page=1
    - comp_id=Page1_Block17, role=body, fileid=FILE_0001_FULLTEXT, page=1
    - comp_id=Page1_Block18, role=body, fileid=FILE_0001_FULLTEXT, page=1
    - comp_id=P

In [16]:
[p.number for p in issue_1.pages]

[1, 2, 3, 4, 5, 6, 7, 8]

In [ ]:
draw_issue_boxes(
    issue=issue_1,
    images_dir=os.path.join(BASE_DIR, issue_1.path),
    save_dir=None,
    show=True,
    draw_regions=True,
    draw_image_cis=True,
    max_width = 500
)